In [14]:
import copy
import math
import random
import numpy as np
from random import randint

In [15]:
class Board:

    def __init__(self):
        self.board = np.full((6, 7), ' ', dtype=str)
        self.empty = 42
        self.record = dict()
        self.record[repr(self)] = 1
        self.state = ' '

    def isLegal(self, isX: bool, move: tuple[int, int]):
        r, c = move[0], move[1] - 1
        if r == 0:
            return ((isX and self.board[5, c] == 'X') or (not isX and self.board[5, c] == 'O'))
        if r == 1:
            return self.board[6 - r, c] == ' '
        return (self.board[7 - r, c] != ' ' and self.board[6 - r, c] == ' ')
    
    def playMove(self, icon: str, move: tuple[int, int]):
        r, c = move[0], move[1] - 1
        if r == 0:
            self.board[1:, c] = self.board[:-1, c]
            self.board[0, c] = ' '
            self.empty += 1
        else:
            self.board[6 - r, c] = icon
            self.empty -= 1
        current_state = repr(self)
        if current_state in self.record:
            if self.record[current_state] == 2:
                self.state = "Game ends on a tie!"
            else:
                self.record[current_state] += 1
        else:
            self.record[current_state] = 1  
        self.checkWin(icon)
        self.checkWin('O' if icon == 'X' else 'X')

    def checkWin(self, icon: str):
        b = (self.board == icon)
        if np.any(b[:, :-3] & b[:, 1:-2] & b[:, 2:-1] & b[:, 3:]):
            self.state = f"Game ends on {icon}'s win!"
            return
        if np.any(b[:-3, :] & b[1:-2, :] & b[2:-1, :] & b[3:, :]):
            self.state = f"Game ends on {icon}'s win!"
            return
        if np.any(b[:-3, :-3] & b[1:-2, 1:-2] & b[2:-1, 2:-1] & b[3:, 3:]):
            self.state = f"Game ends on {icon}'s win!"
            return
        if np.any(b[3:, :-3] & b[2:-1, 1:-2] & b[1:-2, 2:-1] & b[:-3, 3:]):
            self.state = f"Game ends on {icon}'s win!"
            return

    def __str__(self):
        printer = "  -----------------------------\n"
        for i in range(6):
            printer += f"{6 - i} |"
            for j in range(7):
                current = self.board[i, j]
                if current in ('X', 'O'):
                    printer += f" {current} |"
                else:
                    printer += "   |"
            printer += "\n  -----------------------------\n"
        printer += "    1   2   3   4   5   6   7"
        return printer
    
    def __repr__(self):
        return "".join(self.board.ravel())

In [ ]:
class MCTSNode:

    def __init__(self, board, move=None, parent=None):
        self.board = copy.deepcopy(board)
        self.move = move
        self.parent = parent
        self.children = []
        self.wins = 0
        self.visits = 0
        self.untriedMoves = Player(True, "MCTS").getPossibleMoves(self.board)

    def select(self):
        return max(self.children, key=lambda c: (c.wins / c.visits) + 1.41 * math.sqrt(math.log(self.visits) / c.visits))
    
    def expand(self, isX):
        move = self.untriedMoves.pop()
        next_board = copy.deepcopy(self.board)
        icon = 'X' if isX else 'O'
        next_board.playMove(icon, move)
        child_node = MCTSNode(next_board, move=move, parent=self)
        self.children.append(child_node)
        return child_node
    
    def update(self, result):
        self.visits += 1
        self.wins += result
        if self.parent:
            self.parent.update(1 - result)

    def is_fully_expanded(self):
        return len(self.untriedMoves) == 0

    def is_terminal(self):
        return self.board.state != ' '

    def rollout(self, isX):
        tempBoard = copy.deepcopy(self.board)
        currentIsX = isX
        while tempBoard.state == ' ':
            moves = Player(currentIsX, "sim").getPossibleMoves(tempBoard)
            move = random.choice(moves)
            icon = 'X' if currentIsX else 'O'
            tempBoard.playMove(icon, move)
            currentIsX = not currentIsX
        if "X's win" in tempBoard.state: return 1 if isX else 0
        if "O's win" in tempBoard.state: return 0 if isX else 1
        return 0.5

class Player:

    def __init__(self, isX: bool, type: str):
        self.isX = isX
        self.type = type

    def getPossibleMoves(self, board: Board):
        if (board.empty == 0):
            return ["tie"]
        moves = []
        for i in range(1, 8):
            if (board.board[5][i - 1] == 'X' and self.isX) or (board.board[5][i - 1] == 'O' and not self.isX):
                moves.append((0, i))
            for j in range(1, 7):
                if board.board[6 - j][i - 1] == ' ':
                    moves.append((j, i))
                    break
        return moves

    def turn(self, board: Board):
        print(f"{self}'s turn")
        print(f"Possible moves: {self.getPossibleMoves(board)}")
        if self.type == "human":

            while True:
                toParse = input("Enter move coordinates separated by a comma: ").split(",")
                if (board.empty == 0 and len(toParse) == 0 and toParse[0] == "tie"):
                    board.state = "Game ends on a tie!"
                    break
                if len(toParse) != 2:
                    print("Invalid move! Try again.")
                    continue
                move = tuple((int(toParse[0]), int(toParse[1])))
                if move[0] not in range(0, 7) or move[1] not in range(1, 8):
                    print("Invalid move! Try again.")
                    continue
                if board.isLegal(self.isX, move): # type: ignore
                    board.playMove(str(self), move) # type: ignore
                    break
                else:
                    print("Invalid move! Try again.")

        elif self.type == "MCTS":

            def mcts_search(rootBoard: Board, iterations=1000, isX=True):
                rootNode = MCTSNode(rootBoard)
                for _ in range(iterations):
                    node = rootNode
                    currentSimIsX = isX
                    while node.is_fully_expanded() and not node.is_terminal():
                        node = node.select()
                        currentSimIsX = not currentSimIsX
                    if not node.is_terminal():
                        node = node.expand(currentSimIsX)
                        currentSimIsX = not currentSimIsX
                    result = node.rollout(currentSimIsX)
                    node.update(result)
                best_move_node = max(rootNode.children, key=lambda c: c.visits)
                return best_move_node.move
            
            print(NotImplemented)

    def __str__(self):
        if self.isX:
             return 'X'
        return 'O'

In [18]:
def main():
    mode = int(input("Game mode selection\n0 - Human Vs Human\n1 - Human Vs MCTS\n 2 - Human Vs DT\n 3 - MCTS vs DT\nEnter choice: "))
    if mode not in (0, 1, 2):
        return
    if mode == 0:
        playerX = Player(True, "human")
        playerO = Player(False, "human")
    elif mode == 1 or mode == 2:
        if randint(0, 1) == 0:
            playerX = Player(True, "human")
            playerO = Player(False, "MCTS")
        else:
            playerX = Player(False, "human")
            playerO = Player(True, "DT")
    else:
        if randint(0, 1) == 0:
            playerX = Player(True, "MCTS")
            playerO = Player(False, "DT")
        else:
            playerX = Player(True, "DT")
            playerO = Player(False, "MCTS")
    board = Board()
    print(board)
    print("Notes:\nIf you want to pop out, enter coordinates in format:\n0, col\ncol being the number of the column you want to pop out of\nIf the board is full, you can tie by inputing 'tie'")
    while board.state == ' ':
        playerX.turn(board)
        print(board)
        if board.state != ' ':
            break
        playerO.turn(board)
        print(board)
    print(board.state)
        

In [ ]:
main()

  -----------------------------
6 |   |   |   |   |   |   |   |
  -----------------------------
5 |   |   |   |   |   |   |   |
  -----------------------------
4 |   |   |   |   |   |   |   |
  -----------------------------
3 |   |   |   |   |   |   |   |
  -----------------------------
2 |   |   |   |   |   |   |   |
  -----------------------------
1 |   |   |   |   |   |   |   |
  -----------------------------
    1   2   3   4   5   6   7
Notes:
If you want to pop out, enter coordinates in format:
0, col
col being the number of the column you want to pop out of
If the board is full, you can tie by inputing 'tie'
X's turn
Possible moves: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7)]
